<a href="https://colab.research.google.com/github/ggalarza1/accounting/blob/main/unified_accounting_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#!/usr/bin/env python3
"""
UNIFIED ACCOUNTING SYSTEM
Combines Square POS data + Bitcoin wallet data into one comprehensive view
For businesses accepting both traditional payments (Square) and Bitcoin
"""

import json
from datetime import datetime, timedelta
from typing import List, Dict, Optional
from collections import defaultdict

# Import existing modules
from square_accounting import SquareAccountingExtractor
from bitcoin_accounting import generate_comprehensive_report as generate_bitcoin_report
from bitcoin_wallet_extractor import BitcoinWalletExtractor


class UnifiedAccountingSystem:
    """
    Merge Square transactions and Bitcoin transactions into unified financial statements
    """

    def __init__(self, square_token: str = None, square_env: str = 'sandbox'):
        self.square_extractor = None
        if square_token:
            self.square_extractor = SquareAccountingExtractor(square_token, square_env)

        self.bitcoin_extractor = BitcoinWalletExtractor()

        # Storage for unified data
        self.square_data = []
        self.bitcoin_data = {}
        self.unified_transactions = []

    def fetch_square_data(self, start_date: str, end_date: str, location_id: str = None):
        """Pull Square transactions"""
        if not self.square_extractor:
            print("❌ Square not configured. Provide access token.")
            return []

        print(f"\n🔷 Fetching Square data...")
        self.square_data = self.square_extractor.get_payments(start_date, end_date)
        print(f"✅ Retrieved {len(self.square_data)} Square transactions")
        return self.square_data

    def fetch_bitcoin_data(self, address_or_xpub: str, gap_limit: int = 20):
        """Pull Bitcoin wallet data"""
        print(f"\n₿ Fetching Bitcoin data...")

        if self.bitcoin_extractor.is_xpub(address_or_xpub):
            self.bitcoin_data = self.bitcoin_extractor.scan_xpub(address_or_xpub, gap_limit)
        else:
            self.bitcoin_data = self.bitcoin_extractor.get_wallet_summary(address_or_xpub)

        tx_count = self.bitcoin_data.get('transaction_count', 0)
        print(f"✅ Retrieved {tx_count} Bitcoin transactions")
        return self.bitcoin_data

    def normalize_transactions(self) -> List[Dict]:
        """
        Convert both Square and Bitcoin transactions into unified format
        This enables combined analysis and reporting
        """
        unified = []

        # Normalize Square transactions
        for sq_tx in self.square_data:
            unified.append({
                'source': 'square',
                'transaction_id': sq_tx.get('transaction_id'),
                'date': sq_tx.get('date'),
                'datetime': sq_tx.get('datetime'),
                'amount_usd': sq_tx.get('net_amount', 0),
                'gross_amount_usd': sq_tx.get('total_amount', 0),
                'fees_usd': sq_tx.get('processing_fee', 0),
                'currency': 'USD',
                'type': self._classify_square_type(sq_tx),
                'status': sq_tx.get('status'),
                'payment_method': sq_tx.get('source_type', 'CARD'),
                'raw_data': sq_tx
            })

        # Normalize Bitcoin transactions
        btc_txs = self.bitcoin_data.get('transactions', [])
        for btc_tx in btc_txs:
            unified.append({
                'source': 'bitcoin',
                'transaction_id': btc_tx.get('tx_id'),
                'date': btc_tx.get('date_only'),
                'datetime': btc_tx.get('date'),
                'amount_usd': btc_tx.get('net_change_usd', 0) or 0,
                'gross_amount_usd': btc_tx.get('usd_received', 0) or btc_tx.get('usd_sent', 0) or 0,
                'fees_usd': btc_tx.get('usd_fee', 0) or 0,
                'currency': 'BTC',
                'type': self._classify_bitcoin_type(btc_tx),
                'status': 'COMPLETED',
                'payment_method': 'BITCOIN',
                'btc_amount': btc_tx.get('net_change_btc', 0),
                'btc_price': btc_tx.get('usd_price'),
                'realized_gain': btc_tx.get('realized_gain_loss', 0),
                'raw_data': btc_tx
            })

        # Sort by date
        unified.sort(key=lambda x: x.get('date', '') or '9999-12-31')

        self.unified_transactions = unified
        print(f"\n✅ Normalized {len(unified)} total transactions")
        return unified

    def _classify_square_type(self, sq_tx: Dict) -> str:
        """Classify Square transaction for accounting"""
        tx_type = sq_tx.get('transaction_type', 'sale')
        if tx_type in ['refund', 'partial_refund']:
            return 'refund'
        elif tx_type == 'voided':
            return 'void'
        return 'income'

    def _classify_bitcoin_type(self, btc_tx: Dict) -> str:
        """Classify Bitcoin transaction for accounting"""
        tx_type = btc_tx.get('type', 'unknown')
        if tx_type == 'incoming':
            return 'income'
        elif tx_type == 'outgoing':
            # Check if realized gain means it's a sale/d disposition
            if btc_tx.get('realized_gain_loss') is not None:
                return 'sale'
            return 'expense'
        return 'transfer'

    def generate_unified_income_statement(self, start_date: str = None, end_date: str = None) -> Dict:
        """
        Generate Income Statement combining Square + Bitcoin
        """
        txs = self.unified_transactions

        # Filter by date
        if start_date:
            txs = [t for t in txs if t.get('date', '') >= start_date]
        if end_date:
            txs = [t for t in txs if t.get('date', '') <= end_date]

        # Categorize
        revenue = defaultdict(float)
        expenses = defaultdict(float)
        gains_losses = defaultdict(float)

        for tx in txs:
            tx_type = tx.get('type', 'unknown')
            amount = tx.get('amount_usd', 0)
            source = tx.get('source', 'unknown')

            if tx_type == 'income':
                revenue[f'{source.upper()} Revenue'] += abs(amount)

            elif tx_type == 'refund':
                expenses[f'{source.upper()} Refunds'] += abs(amount)

            elif tx_type == 'sale' and source == 'bitcoin':
                # Bitcoin sale with potential gain/loss
                gain = tx.get('realized_gain', 0)
                if gain:
                    gains_losses['Bitcoin Realized Gains' if gain > 0 else 'Bitcoin Realized Losses'] += abs(gain)
                revenue['Bitcoin Sale Proceeds'] += abs(amount)

            elif tx_type == 'expense':
                expenses[f'{source.upper()} Expenses'] += abs(amount)

            # Fees are always expenses
            fees = tx.get('fees_usd', 0)
            if fees > 0:
                expenses[f'{source.upper()} Fees'] += fees

        total_revenue = sum(revenue.values())
        total_expenses = sum(expenses.values())
        net_gains = gains_losses.get('Bitcoin Realized Gains', 0)
        net_losses = gains_losses.get('Bitcoin Realized Losses', 0)

        net_income = total_revenue - total_expenses + net_gains - net_losses

        return {
            'statement_type': 'Unified Income Statement',
            'period': f"{start_date or 'All Time'} to {end_date or 'Present'}",
            'generated_at': datetime.now().isoformat(),
            'revenue': {
                'items': dict(revenue),
                'total': total_revenue
            },
            'expenses': {
                'items': dict(expenses),
                'total': total_expenses
            },
            'gains_losses': {
                'realized_gains': net_gains,
                'realized_losses': net_losses,
                'net': net_gains - net_losses
            },
            'net_income': net_income,
            'payment_breakdown': {
                'square_payments': len([t for t in txs if t.get('source') == 'square']),
                'bitcoin_payments': len([t for t in txs if t.get('source') == 'bitcoin'])
            }
        }

    def generate_unified_balance_sheet(self, as_of_date: str = None) -> Dict:
        """
        Combined Balance Sheet showing both fiat (Square/cash) and crypto holdings
        """
        as_of = as_of_date or datetime.now().strftime('%Y-%m-%d')

        # Square balance (cash position from settlements)
        square_balance = sum(t.get('net_amount', 0) for t in self.square_data
                             if t.get('status') == 'COMPLETED')

        # Bitcoin holdings
        btc_holdings = self.bitcoin_data.get('current_balance_btc', 0)
        btc_price = self.bitcoin_data.get('current_btc_price_usd', 0)
        btc_value_usd = btc_holdings * btc_price
        unrealized_gains = self.bitcoin_data.get('unrealized_gains_losses_usd', 0)

        total_assets = square_balance + btc_value_usd

        return {
            'statement_type': 'Unified Balance Sheet',
            'as_of_date': as_of,
            'generated_at': datetime.now().isoformat(),
            'assets': {
                'fiat_equivalents': {
                    'square_cash_position': square_balance,
                    'currency': 'USD'
                },
                'crypto_assets': {
                    'bitcoin_btc': btc_holdings,
                    'bitcoin_usd': btc_value_usd,
                    'btc_price': btc_price,
                    'unrealized_gain_loss': unrealized_gains
                },
                'total_assets': total_assets
            },
            'summary': {
                'fiat_percentage': (square_balance / total_assets * 100) if total_assets > 0 else 0,
                'crypto_percentage': (btc_value_usd / total_assets * 100) if total_assets > 0 else 0
            }
        }

    def generate_comprehensive_report(self, start_date: str = None, end_date: str = None) -> Dict:
        """
        Generate complete unified report
        """
        return {
            'report_metadata': {
                'generated_at': datetime.now().isoformat(),
                'period': f"{start_date or 'All Time'} to {end_date or 'Present'}",
                'sources': ['square', 'bitcoin']
            },
            'income_statement': self.generate_unified_income_statement(start_date, end_date),
            'balance_sheet': self.generate_unified_balance_sheet(end_date),
            'transaction_summary': {
                'total_transactions': len(self.unified_transactions),
                'by_source': {
                    'square': len([t for t in self.unified_transactions if t.get('source') == 'square']),
                    'bitcoin': len([t for t in self.unified_transactions if t.get('source') == 'bitcoin'])
                }
            }
        }

    def export_unified_csv(self, filename: str = 'unified_transactions.csv'):
        """Export all transactions in unified format"""
        import csv

        fieldnames = [
            'date', 'source', 'transaction_id', 'type', 'amount_usd',
            'fees_usd', 'currency', 'payment_method', 'btc_amount',
            'btc_price', 'realized_gain'
        ]

        with open(filename, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()

            for tx in self.unified_transactions:
                writer.writerow({
                    'date': tx.get('date'),
                    'source': tx.get('source'),
                    'transaction_id': tx.get('transaction_id'),
                    'type': tx.get('type'),
                    'amount_usd': tx.get('amount_usd'),
                    'fees_usd': tx.get('fees_usd'),
                    'currency': tx.get('currency'),
                    'payment_method': tx.get('payment_method'),
                    'btc_amount': tx.get('btc_amount'),
                    'btc_price': tx.get('btc_price'),
                    'realized_gain': tx.get('realized_gain')
                })

        print(f"✅ Exported unified transactions to {filename}")


def print_unified_report(report: Dict):
    """Pretty print the unified financial report"""

    print("\n" + "=" * 80)
    print("💼 UNIFIED ACCOUNTING REPORT")
    print("=" * 80)

    meta = report['report_metadata']
    print(f"Period: {meta['period']}")
    print(f"Sources: {', '.join(meta['sources'])}")
    print(f"Generated: {meta['generated_at'][:19]}")

    # Income Statement
    inc = report['income_statement']
    print("\n" + "-" * 80)
    print("📈 UNIFIED INCOME STATEMENT")
    print("-" * 80)

    print("\n💰 REVENUE:")
    for item, amount in inc['revenue']['items'].items():
        print(f"  {item:.<50} ${amount:>15,.2f}")
    print(f"  {'Total Revenue':.<50} ${inc['revenue']['total']:>15,.2f}")

    print("\n💸 EXPENSES:")
    for item, amount in inc['expenses']['items'].items():
        print(f"  {item:.<50} ${amount:>15,.2f}")
    print(f"  {'Total Expenses':.<50} ${inc['expenses']['total']:>15,.2f}")

    print("\n📊 GAINS/LOSSES:")
    gl = inc['gains_losses']
    print(f"  Realized Gains:   ${gl['realized_gains']:>15,.2f}")
    print(f"  Realized Losses:  ${gl['realized_losses']:>15,.2f}")

    print(f"\n{'NET INCOME':.<50} ${inc['net_income']:>15,.2f}")

    # Payment breakdown
    pb = inc['payment_breakdown']
    print(f"\n📍 PAYMENT METHODS:")
    print(f"  Square Transactions:  {pb}")

ModuleNotFoundError: No module named 'square_accounting'